# Hyperspectral Image Compression on Indian Pines
## Classical codecs vs. neural autoencoders

**Team:** Braj Patel (AU2644047), Dharti Patel (AU2644027), Rudra Patel (AU2644018), Mahi Patel (AU2644046)

This notebook runs the whole study end to end:

1. Load the Indian Pines cube and make a spatial train/test split
2. Define the metrics (PSNR, SAM, bits per pixel per band)
3. Run six classical codecs: PCA, spectral DCT, 3D-DCT, JPEG, JPEG2000, PCA+JPEG2000
4. Train three autoencoders: MLP, 1D-CNN, 3D-CNN
5. Plot rate-distortion curves and build the summary table

Run the cells from top to bottom. Full run on CPU: about 45-90 minutes (the 3D-CNN is the slow part). Set `QUICK = True` in the next cell for a 2-minute smoke test (numbers will be poor).

In [ ]:
# ---- Settings ----
QUICK = False            # True = tiny training runs, just to check everything works

MLP_EPOCHS   = 3 if QUICK else 400
CNN1D_EPOCHS = 3 if QUICK else 300
CNN3D_STEPS  = 5 if QUICK else 1500
SPLIT_COL    = 100       # columns [0, 100) = train, [100, 145) = test
DATA_PATH    = "data/indianpinearray.npy"

import os, io, csv, glob, json
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
os.makedirs("data", exist_ok=True)

## 0. Get the data
Upload `indianpinearray.npy` (the 145 x 145 x 200 Indian Pines cube). On Colab the cell below opens a file picker; locally, just put the file in `data/`.

In [ ]:
if not os.path.exists(DATA_PATH):
    from google.colab import files        # only needed on Colab
    up = files.upload()
    os.replace(list(up.keys())[0], DATA_PATH)
print("Data file found:", os.path.exists(DATA_PATH))

## 1. Spatial train/test split
* The cube is cut vertically at column 100. The left part is the **training region**, the right part the **test region**, so the test set contains fields the models never saw.
* Both parts are min-max normalised with the **training** min and max only, so nothing about the test region leaks into training.

In [ ]:
cube = np.load(DATA_PATH).astype(np.float32)        # (145, 145, 200) = rows, columns, bands
H, W, B = cube.shape
c = SPLIT_COL
train_raw, test_raw = cube[:, :c], cube[:, c:]

lo, hi = float(train_raw.min()), float(train_raw.max())   # train statistics only
train = (train_raw - lo) / (hi - lo)
test  = (test_raw  - lo) / (hi - lo)
print("train", train.shape, "test", test.shape, "norm range", lo, hi)
print("test range after normalisation:", float(test.min()), float(test.max()))

# picture of the split (false colour: bands 50, 27, 17 as R, G, B)
rgb = cube[:, :, [50, 27, 17]]
p1, p99 = np.percentile(rgb, [1, 99])
plt.figure(figsize=(4.5, 4.5))
plt.imshow(np.clip((rgb - p1) / (p99 - p1), 0, 1))
plt.axvline(c - 0.5, color="white", linewidth=2.5)
plt.title("Train (left) / Test (right)"); plt.axis("off"); plt.show()

## 2. Metrics and bit counting
* **PSNR** = 10 log10(1 / MSE) on normalised data. Higher is better.
* **SAM** = mean angle (degrees) between each original and reconstructed spectrum. Lower is better.
* **quantize** turns a latent code into 8-bit numbers and returns how many bits that costs.
* **bpppb** = total bits / (pixels x bands). The raw data is 16 bpppb.

In [ ]:
import numpy as np
import torch

def psnr(x, y, peak=1.0):
    return float(10 * np.log10(peak**2 / np.mean((x - y) ** 2)))

def sam(x, y, eps=1e-8):
    """Mean spectral angle in degrees; x, y shaped (..., bands)."""
    x = x.reshape(-1, x.shape[-1]).astype(np.float64)
    y = y.reshape(-1, y.shape[-1]).astype(np.float64)
    cos = (x * y).sum(1) / (np.linalg.norm(x, axis=1) * np.linalg.norm(y, axis=1) + eps)
    return float(np.degrees(np.arccos(np.clip(cos, -1, 1))).mean())

def quantize(z, bits=8, ch_dim=-1):
    """Uniform per-channel quantization of a latent tensor.
    Each channel gets its own min/max (stored as 2 float32 = side info).
    Returns (dequantized tensor, total bits incl. side info)."""
    z = torch.as_tensor(z, dtype=torch.float32)
    ch_dim = ch_dim % z.dim()
    dims = [d for d in range(z.dim()) if d != ch_dim]
    lo = z.amin(dim=dims, keepdim=True)
    hi = z.amax(dim=dims, keepdim=True)
    L = 2**bits - 1
    q = torch.round((z - lo) / (hi - lo + 1e-12) * L)
    side_bits = 2 * 32 * z.shape[ch_dim]
    return q / L * (hi - lo) + lo, z.numel() * bits + side_bits

def bpppb(total_bits, cube_shape):
    """Bits per pixel per band (original data is 16 bpppb)."""
    return float(total_bits / np.prod(cube_shape))

def score(orig, rec, total_bits):
    return {"bpppb": bpppb(total_bits, orig.shape),
            "PSNR": psnr(orig, rec), "SAM": sam(orig, rec)}

In [ ]:
import csv, os
import numpy as np

class Logger:
    """Collects rows, prints them, writes results/<name>.csv,
    saves one reconstruction per method for the visual figures."""
    def __init__(self, group, csv_name):
        """group = "classical" or "neural" -> results/<group>/<csv_name>.csv"""
        self.dir = f"results/{group}"
        self.path = f"{self.dir}/{csv_name}.csv"
        self.rows = []
        os.makedirs(f"{self.dir}/recon", exist_ok=True)

    def log(self, method, setting, r, rec=None):
        print(f"{method:14s} {setting:>10}  bpppb={r['bpppb']:.3f}  "
              f"PSNR={r['PSNR']:.2f} dB  SAM={r['SAM']:.2f} deg", flush=True)
        self.rows.append([method, setting, r["bpppb"], r["PSNR"], r["SAM"]])
        if rec is not None:
            np.save(f"{self.dir}/recon/{method}.npy", rec.astype(np.float16))
            with open(f"{self.dir}/recon/{method}.txt", "w") as f:
                f.write(f"{r['bpppb']:.4f}")
        self.write()

    def write(self):
        with open(self.path, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["method", "setting", "bpppb", "PSNR", "SAM"])
            w.writerows(self.rows)

## 3. Classical codecs
Each function takes the normalised cube(s) and returns `(reconstruction, total_bits)`.
* **PCA**: basis learned on the train region, top-k components kept.
* **Spectral DCT**: 1D DCT along the bands, keep the k highest-energy coefficients.
* **3D-DCT**: JPEG-style 8x8x8 blocks, uniform quantisation, entropy-estimated rate.
* **JPEG / JPEG2000**: all bands tiled into one mosaic image; rate = real file size.
* **PCA+JPEG2000**: PCA to 20 components, then JPEG2000 on the component images.

In [ ]:
"""Classical baselines. All take normalized [0,1] cubes (H, W, B) and
return (reconstruction, total_bits)."""
import io
import numpy as np
from PIL import Image
from scipy.fft import dct, idct, dctn, idctn
from sklearn.decomposition import PCA


# ---------- spectral transforms + uniform quantization ----------

def pca_codec(train, test, k, bits=8):
    """PCA basis learned on TRAIN region, applied to TEST."""
    B = test.shape[-1]
    pca = PCA(k).fit(train.reshape(-1, B))
    z = pca.transform(test.reshape(-1, B))
    zq, nbits = quantize(z, bits)
    rec = pca.inverse_transform(zq.numpy()).reshape(test.shape)
    return np.clip(rec, 0, 1), nbits


def dct_codec(train, test, k, bits=8):
    """Spectral 1D DCT + zonal coding: keep the k DCT coefficients with the
    highest energy on the TRAIN region, 8-bit quantize them."""
    B = test.shape[-1]
    energy = (dct(train.reshape(-1, B), norm="ortho", axis=1) ** 2).mean(0)
    keep = np.argsort(-energy)[:k]
    c = dct(test.reshape(-1, B), norm="ortho", axis=1)[:, keep]
    cq, nbits = quantize(c, bits)
    full = np.zeros((c.shape[0], B), np.float32)
    full[:, keep] = cq.numpy()
    rec = idct(full, norm="ortho", axis=1).reshape(test.shape)
    return np.clip(rec, 0, 1), nbits


def _entropy_bits(q):
    """Zero-order entropy, one adaptive model per coefficient position
    (an estimate of what an arithmetic coder would spend)."""
    total = 0.0
    for pos in range(q.shape[1]):
        _, cnt = np.unique(q[:, pos], return_counts=True)
        total += -(cnt * np.log2(cnt / cnt.sum())).sum()
    return total


def dct3d_codec(test, step, b=8):
    """JPEG-style 3D DCT on 8x8x8 blocks, uniform quantization with
    step size `step`, rate = entropy estimate of the quantized coefficients."""
    H, W, B = test.shape
    Hp, Wp = -(-H // b) * b, -(-W // b) * b
    x = np.pad(test, ((0, Hp - H), (0, Wp - W), (0, 0)), mode="edge")
    blk = (x.reshape(Hp//b, b, Wp//b, b, B//b, b)
            .transpose(0, 2, 4, 1, 3, 5).reshape(-1, b, b, b))
    q = np.round(dctn(blk, axes=(1, 2, 3), norm="ortho") / step)
    bits = _entropy_bits(q.reshape(len(q), -1))
    R = idctn(q * step, axes=(1, 2, 3), norm="ortho")
    R = (R.reshape(Hp//b, Wp//b, B//b, b, b, b)
          .transpose(0, 3, 1, 4, 2, 5).reshape(Hp, Wp, B)[:H, :W])
    return np.clip(R, 0, 1), bits


# ---------- image codecs ----------
# Bands are tiled into ONE mosaic image so the file header is paid once,
# not 200 times (otherwise header overhead dominates at these small sizes).

def _mosaic(cube, cols):
    H, W, B = cube.shape
    rows = int(np.ceil(B / cols))
    m = np.zeros((rows * H, cols * W), cube.dtype)
    for b in range(B):
        r, c = divmod(b, cols)
        m[r*H:(r+1)*H, c*W:(c+1)*W] = cube[:, :, b]
    return m

def _unmosaic(m, shape, cols):
    H, W, B = shape
    out = np.zeros(shape, np.float32)
    for b in range(B):
        r, c = divmod(b, cols)
        out[:, :, b] = m[r*H:(r+1)*H, c*W:(c+1)*W]
    return out


def jpeg_codec(test, quality, cols=20):
    """Baseline JPEG (8-bit) on a mosaic of all bands."""
    m = _mosaic(np.round(np.clip(test, 0, 1) * 255).astype(np.uint8), cols)
    buf = io.BytesIO()
    Image.fromarray(m).save(buf, "JPEG", quality=quality)
    dec = np.array(Image.open(io.BytesIO(buf.getvalue())), np.float32) / 255
    return _unmosaic(dec, test.shape, cols), len(buf.getvalue()) * 8


def _j2k(img16, ratio):
    buf = io.BytesIO()
    Image.fromarray(img16).save(buf, "JPEG2000", quality_mode="rates",
                                quality_layers=[ratio], irreversible=True)
    dec = np.array(Image.open(io.BytesIO(buf.getvalue())), np.float32)
    return dec, len(buf.getvalue()) * 8


def jpeg2000_codec(test, target_bpppb, cols=20):
    """JPEG2000 (16-bit) on a mosaic of all bands."""
    m = _mosaic(np.round(np.clip(test, 0, 1) * 65535).astype(np.uint16), cols)
    dec, nbits = _j2k(m, 16 / target_bpppb)
    return _unmosaic(dec / 65535, test.shape, cols), nbits


def pca_jpeg2000_codec(train, test, k, target_bpppb):
    """Standard HSI pipeline: PCA across bands (basis from TRAIN),
    then JPEG2000 on the k principal-component images."""
    H, W, B = test.shape
    pca = PCA(k).fit(train.reshape(-1, B))
    z = pca.transform(test.reshape(-1, B)).reshape(H, W, k)
    lo, hi = z.min((0, 1)), z.max((0, 1))
    zn = (z - lo) / (hi - lo)
    m = _mosaic(np.round(zn * 65535).astype(np.uint16), cols=k)
    budget = target_bpppb * H * W * B
    dec, nbits = _j2k(m, (H * W * k * 16) / budget)
    zr = _unmosaic(dec / 65535, (H, W, k), cols=k) * (hi - lo) + lo
    rec = pca.inverse_transform(zr.reshape(-1, k)).reshape(test.shape)
    side = 2 * 32 * k
    return np.clip(rec, 0, 1), nbits + side

In [ ]:
L = Logger("classical", "classical")

def run(method, setting, out, save):
    rec, bits = out
    L.log(method, setting, score(test, rec, bits), rec if save else None)

for k in [2, 4, 8, 16, 32]:
    run("PCA", f"k={k}", pca_codec(train, test, k), k == 4)
for k in [4, 8, 16, 32, 64]:
    run("DCT", f"k={k}", dct_codec(train, test, k), k == 4)
for s in [0.3, 0.2, 0.15, 0.1, 0.05, 0.03, 0.02]:
    run("3D-DCT", f"step={s}", dct3d_codec(test, s), s == 0.15)
for q in [5, 10, 25, 50, 75, 90]:
    run("JPEG", f"q={q}", jpeg_codec(test, q), q == 10)
for b in [0.05, 0.1, 0.16, 0.2, 0.4, 0.8, 1.6]:
    run("JPEG2000", f"{b}bpppb", jpeg2000_codec(test, b), b == 0.16)
for b in [0.02, 0.05, 0.1, 0.16, 0.2, 0.4, 0.8]:
    run("PCA+JPEG2000", f"{b}bpppb", pca_jpeg2000_codec(train, test, 20, b), b == 0.16)

## 4. Neural autoencoders
### 4.1 Training and evaluation loops
* Loss: MSE. Optimiser: Adam, learning rate 2e-3, cosine schedule.
* At test time the latent code is quantised to 8 bits **before** decoding, so the reported PSNR includes the quantisation error.

In [ ]:
import numpy as np
import torch
import torch.nn as nn


# ---------- per-pixel models (Model 1 MLP, Model 2 1D-CNN) ----------

def train_model(model, X, epochs=400, batch=256, lr=2e-3, log_every=100):
    """X: (num_pixels, bands) training spectra."""
    X = torch.tensor(X)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    loss_fn = nn.MSELoss()
    model.train()
    for ep in range(epochs):
        perm = torch.randperm(len(X))
        total = 0.0
        for i in range(0, len(X), batch):
            xb = X[perm[i:i + batch]]
            loss = loss_fn(model(xb), xb)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(xb)
        sched.step()
        if log_every and (ep % log_every == 0 or ep == epochs - 1):
            print(f"  epoch {ep:4d}  train MSE {total / len(X):.6f}", flush=True)
    return model


@torch.no_grad()
def evaluate(model, test, bits=8):
    """test: (H, W, bands) cube. Returns (metrics dict, reconstruction)."""
    model.eval()
    z = model.encode(torch.tensor(test.reshape(-1, test.shape[-1])))
    zq, nbits = quantize(z, bits, ch_dim=-1)
    rec = model.decode(zq).numpy().reshape(test.shape)
    return score(test, rec, nbits), rec


# ---------- patch model (Model 3 3D-CNN) ----------

def to_vol(cube):
    """(H, W, B) numpy -> (1, 1, B, H, W) tensor."""
    return torch.tensor(cube).permute(2, 0, 1)[None, None]


def train_patch_model(model, train, steps=1500, batch=4, patch=24, lr=2e-3,
                      log_every=250):
    """Trains on random spatial crops of the TRAIN region."""
    H, W, _ = train.shape
    vol = to_vol(train)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    loss_fn = nn.MSELoss()
    model.train()
    for step in range(steps):
        ys = np.random.randint(0, H - patch + 1, batch)
        xs = np.random.randint(0, W - patch + 1, batch)
        xb = torch.cat([vol[..., y:y + patch, x:x + patch] for y, x in zip(ys, xs)])
        loss = loss_fn(model(xb), xb)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if log_every and (step % log_every == 0 or step == steps - 1):
            print(f"  step {step:5d}  train MSE {loss.item():.6f}", flush=True)
    return model


@torch.no_grad()
def evaluate_patch(model, test, bits=8, mult=4):
    """Whole test region in one pass (padded to a multiple of `mult`).
    Bits are counted on the padded code, so padding overhead is included."""
    model.eval()
    H, W, _ = test.shape
    ph, pw = (-H) % mult, (-W) % mult
    padded = np.pad(test, ((0, ph), (0, pw), (0, 0)), mode="reflect")
    z = model.encode(to_vol(padded))
    zq, nbits = quantize(z, bits, ch_dim=1)
    rec = model.decode(zq)[0, 0].permute(1, 2, 0).numpy()[:H, :W]
    return score(test, rec, nbits), rec

### 4.2 Model 1: spectral MLP autoencoder
Per pixel: 200 -> 128 -> 64 -> k -> 64 -> 128 -> 200. A k-number code at 8 bits = 0.04k bpppb.

In [ ]:
import torch.nn as nn

class MLPAE(nn.Module):
    """Model 1: per-pixel spectral MLP autoencoder."""
    def __init__(self, bands=200, k=16):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(bands, 128), nn.ReLU(),
                                 nn.Linear(128, 64), nn.ReLU(),
                                 nn.Linear(64, k))
        self.dec = nn.Sequential(nn.Linear(k, 64), nn.ReLU(),
                                 nn.Linear(64, 128), nn.ReLU(),
                                 nn.Linear(128, bands), nn.Sigmoid())
    def encode(self, x): return self.enc(x)
    def decode(self, z): return self.dec(z)
    def forward(self, x): return self.decode(self.encode(x))

In [ ]:
import torch

EPOCHS = MLP_EPOCHS
torch.manual_seed(0)
L = Logger("neural", "mlp_ae")
for k in [4, 8, 16]:
    print(f"Training MLP-AE k={k}")
    m = train_model(MLPAE(k=k), train.reshape(-1, 200), epochs=EPOCHS)
    r, rec = evaluate(m, test)
    L.log("MLP-AE", f"k={k}", r, rec if k == 4 else None)

### 4.3 Model 2: spectral 1D-CNN autoencoder
Per pixel, but with convolutions sliding along the spectrum (200 -> 100 -> 50 -> 25 samples), then a linear layer to k.

In [ ]:
import torch.nn as nn

class CNN1DAE(nn.Module):
    """Model 2: per-pixel 1D-CNN autoencoder.
    Convolutions slide along the spectrum (neighbouring bands share filters),
    then a linear layer squeezes the features into a k-dim code."""
    def __init__(self, bands=200, k=16):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv1d(1, 16, 7, stride=2, padding=3), nn.ReLU(),   # 200 -> 100
            nn.Conv1d(16, 32, 5, stride=2, padding=2), nn.ReLU(),  # 100 -> 50
            nn.Conv1d(32, 32, 5, stride=2, padding=2), nn.ReLU(),  # 50 -> 25
            nn.Flatten(), nn.Linear(32 * 25, k))
        self.dec_fc = nn.Sequential(nn.Linear(k, 32 * 25), nn.ReLU())
        self.dec = nn.Sequential(
            nn.ConvTranspose1d(32, 32, 4, stride=2, padding=1), nn.ReLU(),  # 25 -> 50
            nn.ConvTranspose1d(32, 16, 4, stride=2, padding=1), nn.ReLU(),  # 50 -> 100
            nn.ConvTranspose1d(16, 1, 4, stride=2, padding=1), nn.Sigmoid())  # 100 -> 200
    def encode(self, x):            # x: (N, 200)
        return self.enc(x.unsqueeze(1))
    def decode(self, z):            # z: (N, k)
        return self.dec(self.dec_fc(z).view(-1, 32, 25)).squeeze(1)
    def forward(self, x):
        return self.decode(self.encode(x))

In [ ]:
import torch

EPOCHS = CNN1D_EPOCHS
torch.manual_seed(0)
L = Logger("neural", "cnn1d_ae")
for k in [4, 8, 16]:
    print(f"Training 1D-CNN-AE k={k}")
    m = train_model(CNN1DAE(k=k), train.reshape(-1, 200), epochs=EPOCHS)
    r, rec = evaluate(m, test)
    L.log("1D-CNN-AE", f"k={k}", r, rec if k == 4 else None)

### 4.4 Model 3: spatial-spectral 3D-CNN autoencoder
Convolves over bands, rows and columns together. Trained on random 24x24 crops of the train region and applied to the whole test region in one pass. **Note:** with 1,500 CPU steps this model is undertrained (train PSNR ~ test PSNR, loss still falling); see the report.

In [ ]:
import torch.nn as nn

def block(cin, cout, k, s):
    pad = tuple(kk // 2 for kk in k)
    return nn.Sequential(nn.Conv3d(cin, cout, k, stride=s, padding=pad), nn.ReLU())

class CNN3DAE(nn.Module):
    """Model 3: fully convolutional 3D autoencoder on (bands, H, W) volumes.
    Uses spatial AND spectral redundancy. Downsamples bands x8, space x4,
    so the code is c x 25 x H/4 x W/4  ->  bpppb = c * 0.0625 at 8 bits.
    Input H, W must be multiples of 4."""
    def __init__(self, c=4):
        super().__init__()
        self.enc = nn.Sequential(
            block(1, 8, (7, 3, 3), (2, 1, 1)),    # D 200 -> 100
            block(8, 16, (5, 3, 3), (2, 2, 2)),   # D 100 -> 50,  H,W /2
            block(16, 16, (5, 3, 3), (2, 2, 2)),  # D 50 -> 25,   H,W /4
            nn.Conv3d(16, c, 1))
        self.dec = nn.Sequential(
            nn.Conv3d(c, 16, 1), nn.ReLU(),
            nn.Upsample(scale_factor=(2, 2, 2)), block(16, 16, (5, 3, 3), 1),
            nn.Upsample(scale_factor=(2, 2, 2)), block(16, 8, (5, 3, 3), 1),
            nn.Upsample(scale_factor=(2, 1, 1)),
            nn.Conv3d(8, 1, (7, 3, 3), padding=(3, 1, 1)), nn.Sigmoid())
    def encode(self, x):            # x: (N, 1, 200, H, W)
        return self.enc(x)
    def decode(self, z):
        return self.dec(z)
    def forward(self, x):
        return self.decode(self.encode(x))

In [ ]:
import numpy as np, torch

STEPS = CNN3D_STEPS
torch.manual_seed(0); np.random.seed(0)
L = Logger("neural", "cnn3d_ae")
for c in [1, 2, 3, 4]:
    print(f"Training 3D-CNN-AE c={c}")
    m = train_patch_model(CNN3DAE(c=c), train, steps=STEPS)
    r, rec = evaluate_patch(m, test)
    L.log("3D-CNN-AE", f"c={c}", r, rec if c == 3 else None)

## 5. Results: rate-distortion curves, summary table, visual comparison
Reads the CSV files written above and draws all figures (also saved to `results/figures/`).

In [ ]:
"""Reads results/classical + results/neural (CSVs and saved reconstructions), writes figures + summary table
to results/figures/. Run after the run_*.py scripts."""
import glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT = "results/figures"
os.makedirs(OUT, exist_ok=True)

# Fixed colour per method (colour follows the method in every figure).
COLORS = {
    "PCA": "#2a78d6", "DCT": "#eb6834", "3D-DCT": "#1baf7a",
    "JPEG": "#eda100", "JPEG2000": "#e87ba4", "PCA+JPEG2000": "#008300",
    "MLP-AE": "#6250d6", "1D-CNN-AE": "#e34948", "3D-CNN-AE": "#8c8c8c",
}
MARKERS = {"PCA": "o", "DCT": "s", "3D-DCT": "D", "JPEG": "^", "JPEG2000": "v",
           "PCA+JPEG2000": "P", "MLP-AE": "o", "1D-CNN-AE": "s", "3D-CNN-AE": "D"}
NEURAL = ["MLP-AE", "1D-CNN-AE", "3D-CNN-AE"]
CLASSICAL = ["PCA", "DCT", "3D-DCT", "JPEG", "JPEG2000", "PCA+JPEG2000"]

plt.rcParams.update({
    "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "axes.grid": True, "grid.color": "#e1e0d9",
    "grid.linewidth": 0.6, "legend.frameon": False, "savefig.dpi": 200,
    "savefig.bbox": "tight"})

df = pd.concat([pd.read_csv(f) for f in
                sorted(glob.glob("results/classical/*.csv") + glob.glob("results/neural/*.csv"))])
df = df.sort_values(["method", "bpppb"])


def rd_figure(methods, fname, title, xmax=1.7):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    for m in methods:
        d = df[(df.method == m) & (df.bpppb <= xmax)]
        if d.empty:
            continue
        neural = m in NEURAL
        kw = dict(color=COLORS[m], marker=MARKERS[m], markersize=7, linewidth=2,
                  linestyle="-" if neural else "--",
                  markerfacecolor=COLORS[m] if neural else "white",
                  markeredgewidth=1.6, label=m)
        axes[0].plot(d.bpppb, d.PSNR, **kw)
        axes[1].plot(d.bpppb, d.SAM, **kw)
    axes[0].set(xlabel="Rate (bits per pixel per band)", ylabel="PSNR (dB)  ↑ better")
    axes[1].set(xlabel="Rate (bits per pixel per band)", ylabel="SAM (degrees)  ↓ better")
    axes[1].legend(loc="upper right", ncol=1)
    fig.suptitle(title, fontsize=13)
    fig.savefig(f"{OUT}/{fname}")
    plt.show()


rd_figure(CLASSICAL, "fig1_classical_rate_distortion.png",
          "Classical codecs on the held-out test region")
rd_figure(NEURAL + ["PCA", "JPEG2000", "PCA+JPEG2000"],
          "fig2_neural_vs_classical.png",
          "Neural autoencoders vs. strongest classical codecs", xmax=0.8)
rd_figure(CLASSICAL + NEURAL, "fig0_all_methods.png",
          "All methods (dashed = classical, solid = neural)")


# ---------- summary table: PSNR / SAM interpolated at fixed rates ----------
RATES = [0.1, 0.2, 0.3, 0.5]
rows = []
for m in CLASSICAL + NEURAL:
    d = df[df.method == m].sort_values("bpppb")
    if d.empty:
        continue
    row = {"method": m}
    for r in RATES:
        ok = d.bpppb.min() <= r <= d.bpppb.max()
        lr, lb = np.log(r), np.log(d.bpppb.values)
        row[f"PSNR@{r}"] = round(float(np.interp(lr, lb, d.PSNR)), 2) if ok else None
        row[f"SAM@{r}"] = round(float(np.interp(lr, lb, d.SAM)), 2) if ok else None
    rows.append(row)
table = pd.DataFrame(rows)
table.to_csv(f"{OUT}/summary_table.csv", index=False)
with open(f"{OUT}/summary_table.md", "w") as f:
    f.write("PSNR (dB) / SAM (deg) at fixed rates, interpolated in log-rate. "
            "Blank = method's tested range does not reach that rate.\n\n")
    f.write("| Method | " + " | ".join(f"{r} bpppb" for r in RATES) + " |\n")
    f.write("|---" * (len(RATES) + 1) + "|\n")
    for row in rows:
        cells = []
        for r in RATES:
            p, s = row[f"PSNR@{r}"], row[f"SAM@{r}"]
            cells.append("" if p is None else f"{p:.2f} / {s:.2f}")
        f.write(f"| {row['method']} | " + " | ".join(cells) + " |\n")
print(open(f"{OUT}/summary_table.md").read())


# ---------- visual comparison ----------
RGB = [50, 27, 17]   # common false-colour band choice for Indian Pines

def false_color(cube):
    img = cube[:, :, RGB].astype(np.float32)
    lo, hi = np.percentile(test[:, :, RGB], [1, 99])
    return np.clip((img - lo) / (hi - lo), 0, 1)

recon_files = glob.glob("results/classical/recon/*.npy") + glob.glob("results/neural/recon/*.npy")
recs = {os.path.basename(p)[:-4]: np.load(p).astype(np.float32) for p in recon_files}
rate_file = {os.path.basename(p)[:-4]: p[:-4] + ".txt" for p in recon_files}
order = [m for m in ["PCA", "3D-DCT", "JPEG", "JPEG2000", "PCA+JPEG2000",
                     "MLP-AE", "1D-CNN-AE", "3D-CNN-AE"] if m in recs]
saved_rate = {m: float(open(rate_file[m]).read()) for m in recs}

n = len(order) + 1
fig, axes = plt.subplots(2, n, figsize=(1.45 * n, 7.4))
axes[0, 0].imshow(false_color(test)); axes[0, 0].set_title("Original\n16 bpppb", fontsize=8)
axes[1, 0].axis("off")
vmax = 0.05
for i, m in enumerate(order, start=1):
    axes[0, i].imshow(false_color(recs[m]))
    axes[0, i].set_title(f"{m}\n{saved_rate[m]:.2f} bpppb", fontsize=8)
    e = np.sqrt(np.mean((recs[m] - test) ** 2, axis=2))
    im = axes[1, i].imshow(e, cmap="Blues", vmin=0, vmax=vmax)
    axes[1, i].set_title("error", fontsize=8)
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
cbar = fig.colorbar(im, ax=axes[1, 1:], orientation="horizontal", fraction=0.05, pad=0.04)
cbar.set_label("Per-pixel RMSE (normalized units)")
fig.suptitle("Test region at ~0.16 bpppb (100x compression): false colour (bands 50/27/17) and per-pixel error", fontsize=12)
fig.savefig(f"{OUT}/fig3_visual_comparison.png")
plt.show()


# ---------- spectrum of one pixel ----------
y, x = 70, 20
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.plot(test[y, x], color="#0b0b0b", linewidth=2.4, label="Original")
for m in ["PCA", "JPEG2000", "MLP-AE", "3D-CNN-AE"]:
    if m in recs:
        ax.plot(recs[m][y, x], color=COLORS[m], linewidth=1.6,
                linestyle="-" if m in NEURAL else "--",
                label=f"{m} ({saved_rate[m]:.2f} bpppb)")
ax.set(xlabel="Band index", ylabel="Normalized reflectance",
       title=f"Reconstructed spectrum of test pixel (row {y}, col {x})")
ax.legend(loc="upper right")
fig.savefig(f"{OUT}/fig4_spectrum.png")
plt.show()
print("Figures written to", OUT)